# rsvd SVD computation — recount2 subsamples

Computes SVD using `rsvd::rsvd()` for the same subsampled datasets as
`07_recount2_coverage_multiplier_imp_fixed.ipynb` (which uses `big_randomSVD`).

- Reads `subsample_info.rds` from each run directory to recover the sample indices.
- Extracts the submatrix from the cached preprocessed FBM.
- Runs `rsvd::rsvd(Y_sub, k = SVD_K)` with the same `k` as `big_randomSVD`.
- Saves the result as `svd_rsvd.rds` alongside `svd.rds` in each run directory.

Output used by `11_curvature_num_pc.ipynb` to compare elbow / Gavish-Donoho
estimates between the two SVD implementations.

## Load libraries

In [1]:
start_time <- Sys.time()
cat("rsvd SVD computation started at:", format(start_time), "\n")

rsvd SVD computation started at: 2026-02-27 14:47:33 


In [2]:
library(bigstatsr)
library(rsvd)
library(here)
library(dplyr)

source(here("config.R"))

here() starts at /home/msubirana/Documents/pivlab/clamp-analyses


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




## Configuration

In [3]:
output_dir   <- file.path(config$GENERAL$OUTPUT_DIR, "recount2_multiplier_imp_fixed")
sample_sizes <- c(500, 1000, 2000, 4000, 8000, 16000, 32000)
n_seeds      <- 3

message("Output dir   : ", output_dir)
message("Sample sizes : ", paste(sample_sizes, collapse = ", "))
message("Seeds per size: ", n_seeds)

Output dir   : /home/msubirana/Documents/pivlab/clamp-analyses/output/recount2_multiplier_imp_fixed

Sample sizes : 500, 1000, 2000, 4000, 8000, 16000, 32000

Seeds per size: 3



## Load preprocessed FBM

Reuses the cached preprocessed FBM created by
`07_recount2_coverage_multiplier_imp_fixed.ipynb`.

In [4]:
preproc_fbm_rds <- file.path(output_dir, "FBMrecount2_cov_preproc_filtered.rds")

if (!file.exists(preproc_fbm_rds)) {
    stop("Preprocessed FBM not found. Run 07_recount2_coverage_multiplier_imp_fixed.ipynb first.")
}

recount2_fbm_filt <- readRDS(preproc_fbm_rds)
n_genes       <- nrow(recount2_fbm_filt)
n_samps_total <- ncol(recount2_fbm_filt)
message("Loaded preprocessed FBM: ", n_genes, " genes x ", n_samps_total, " samples")

Loaded preprocessed FBM: 6000 genes x 37032 samples



## Compute rsvd for all subsamples

For each run, the sample indices are read from `subsample_info.rds`.
The submatrix is extracted in memory and passed to `rsvd::rsvd()`.
Skips any run where `svd_rsvd.rds` already exists.

In [5]:
for (n_target in sample_sizes) {
    for (run_idx in seq_len(n_seeds)) {

        run_dir <- file.path(
            output_dir,
            paste0("c2cp_subsample_", n_target, "_seed_", run_idx)
        )

        if (!dir.exists(run_dir)) {
            message("Skipping (not found): ", run_dir)
            next
        }

        out_rds <- file.path(run_dir, "svd_rsvd.rds")
        if (file.exists(out_rds)) {
            message(sprintf("Already exists — skipping n=%d run=%d", n_target, run_idx))
            next
        }

        message(sprintf("\n── n=%d  run=%d ──────────────────────────────", n_target, run_idx))

        # ── Recover sample indices ────────────────────────────────────────
        info       <- readRDS(file.path(run_dir, "subsample_info.rds"))
        sample_idx <- info$sample_idx
        n_samples  <- info$n_samples
        message("  n_samples : ", n_samples)

        # ── Extract submatrix from FBM ────────────────────────────────────
        message("  Extracting submatrix from FBM...")
        Y_sub <- recount2_fbm_filt[, sample_idx]   # regular R matrix (genes x samples)

        # ── Compute rsvd ──────────────────────────────────────────────────
        SVD_K <- min(n_samples - 1L, n_genes - 1L)
        message(sprintf("  Computing rsvd (k=%d)...", SVD_K))

        t_svd <- system.time({
            svd_rsvd <- rsvd::rsvd(Y_sub, k = SVD_K)
        })["elapsed"]
        message(sprintf("  rsvd done in %.1fs", t_svd))

        # ── Filter NaN components ─────────────────────────────────────────
        valid_idx        <- which(!is.nan(svd_rsvd$d))
        svd_rsvd$d       <- svd_rsvd$d[valid_idx]
        svd_rsvd$u       <- svd_rsvd$u[, valid_idx, drop = FALSE]
        svd_rsvd$v       <- svd_rsvd$v[, valid_idx, drop = FALSE]

        saveRDS(svd_rsvd, out_rds)
        message("  Saved: ", out_rds)

        rm(Y_sub, svd_rsvd)
        gc()
    }
}

message("\nAll rsvd SVDs computed.")


── n=500  run=1 ──────────────────────────────

  n_samples : 500

  Extracting submatrix from FBM...

  Computing rsvd (k=499)...

  rsvd done in 3.3s

  Saved: /home/msubirana/Documents/pivlab/clamp-analyses/output/recount2_multiplier_imp_fixed/c2cp_subsample_500_seed_1/svd_rsvd.rds


── n=500  run=2 ──────────────────────────────

  n_samples : 500

  Extracting submatrix from FBM...

  Computing rsvd (k=499)...

  rsvd done in 2.7s

  Saved: /home/msubirana/Documents/pivlab/clamp-analyses/output/recount2_multiplier_imp_fixed/c2cp_subsample_500_seed_2/svd_rsvd.rds


── n=500  run=3 ──────────────────────────────

  n_samples : 500

  Extracting submatrix from FBM...

  Computing rsvd (k=499)...

  rsvd done in 2.4s

  Saved: /home/msubirana/Documents/pivlab/clamp-analyses/output/recount2_multiplier_imp_fixed/c2cp_subsample_500_seed_3/svd_rsvd.rds


── n=1000  run=1 ──────────────────────────────

  n_samples : 1000

  Extracting submatrix from FBM...

  Computing rsvd (k=999)...

 

In [6]:
end_time     <- Sys.time()
elapsed_time <- end_time - start_time
cat("Completed at:", format(end_time), "\n")
cat("Elapsed     :", format(elapsed_time), "\n")

Completed at: 2026-02-27 19:27:19 
Elapsed     : 4.662811 hours 
